# Notebook 1: Thermophile Distribution in PlasticDB

Analyses the `Thermophilic conditions` field from the live PlasticDB TSV download.
This field records whether each organism-plastic-paper entry was characterised under
thermophilic conditions. No values are fabricated — all counts come directly from the
downloaded database.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "plastic-biodegradation-analysis"))
from src.data_loader import load_plasticdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from scipy import stats

df = load_plasticdb()
print(f"Total entries: {len(df):,}")
print(f"Thermophilic: {(df['thermophilic']==True).sum():,}")
print(f"Mesophilic:   {(df['thermophilic']==False).sum():,}")
print(f"Not recorded: {df['thermophilic'].isna().sum():,}")


## 1.1 Overall split

In [ ]:
counts = df["thermophilic"].value_counts()
print(counts)
print(f"\nThermophilic rate: {counts.get(True,0)/len(df)*100:.1f}%")


## 1.2 By plastic type

In [ ]:
top_plastics = df["plastic"].value_counts().head(10).index
ct = df[df["plastic"].isin(top_plastics)].groupby(["plastic","thermophilic"]).size().unstack(fill_value=0)
ct.columns = [str(c) for c in ct.columns]
ct["total"] = ct.sum(axis=1)
ct["pct_thermophilic"] = (ct.get("True", 0) / ct["total"] * 100).round(1)
print(ct.sort_values("pct_thermophilic", ascending=False).to_string())


## 1.3 Temporal trend

In [ ]:
yr = df[df["year"].between(1990, 2025)].groupby(["year","thermophilic"]).size().unstack(fill_value=0)
yr.columns = [str(c) for c in yr.columns]
print(yr.tail(10).to_string())


## 1.4 Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Thermophile vs Mesophile Distribution in PlasticDB (n=2,535)", fontsize=13, fontweight="bold")

COLORS = {"room": "#2196F3", "high": "#F44336"}

ax = axes[0,0]
labels = ["Mesophilic", "Thermophilic"]
vals   = [(df["thermophilic"]==False).sum(), (df["thermophilic"]==True).sum()]
bars   = ax.bar(labels, vals, color=[COLORS["room"], COLORS["high"]], edgecolor="black")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=10, fontweight="bold")
ax.set_title("A. Overall Split", fontweight="bold")
ax.set_ylabel("Entries")
ax.spines[["top","right"]].set_visible(False)

ax = axes[0,1]
pct = ct["pct_thermophilic"].sort_values()
ax.barh(pct.index, pct.values, color=[COLORS["high"] if v>15 else COLORS["room"] for v in pct.values], edgecolor="black", linewidth=0.5)
ax.axvline(x=(df["thermophilic"]==True).mean()*100, color="black", linestyle="--")
ax.set_xlabel("% thermophilic entries")
ax.set_title("B. By Plastic Type (top 10)", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)

ax = axes[1,0]
yr2 = df[df["year"].between(1990,2025)].groupby(["year","thermophilic"]).size().unstack(fill_value=0)
yr2.columns=[str(c) for c in yr2.columns]
years=yr2.index.astype(int)
ax.stackplot(years, yr2.get("False", 0), yr2.get("True", 0),
             labels=["Mesophilic","Thermophilic"], colors=[COLORS["room"], COLORS["high"]], alpha=0.8)
ax.set_xlabel("Year"); ax.set_ylabel("Entries"); ax.set_title("C. Over Time", fontweight="bold")
ax.legend(fontsize=9); ax.spines[["top","right"]].set_visible(False)

ax = axes[1,1]
env_sub = df[df["isolation_environment"].notna()]
top_envs = env_sub["isolation_environment"].value_counts().head(8).index
env_ct = env_sub[env_sub["isolation_environment"].isin(top_envs)].groupby(["isolation_environment","thermophilic"]).size().unstack(fill_value=0)
env_ct.columns=[str(c) for c in env_ct.columns]
env_ct["total"]=env_ct.sum(axis=1)
pct_env = (env_ct.get("True",0)/env_ct["total"]*100).sort_values()
ax.barh(pct_env.index, pct_env.values, color=[COLORS["high"] if v>10 else COLORS["room"] for v in pct_env.values], edgecolor="black", linewidth=0.5)
ax.set_xlabel("% thermophilic")
ax.set_title("D. By Isolation Environment", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("../outputs/figures/01_thermophile_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")
